# Numerical Integration Workbook

Melanie Zaidel. File created 2/18/2026 for the Polaris Mentorship Course.

**Objectives:**
- Learn how to implement a simple numerical integrator (trapezoidal rule).
- Compare your implementation with SciPy's `quad` integrator.
- Integrate the NFW Profile to determine the mass of dark matter in the Milky Way.

## Setup

This notebook uses `numpy`, `matplotlib`, and `scipy`. If they are not installed, run the cell below once to install them.

In [ ]:
# Uncomment and run if packages are missing
# !pip install numpy matplotlib scipy --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate
%matplotlib inline

## Section 1 — Integrate `f(x) = x**2` on [0, 1]

First, let's plot it.

In [ ]:
# Plot the function f(x) = x^2 on [0, 1]
x = np.linspace(a, b, 100)
y = x**2

plt.figure(figsize=(6, 4))
plt.plot(x, y, 'b-', linewidth=2)
plt.fill_between(x, y, alpha=0.3)
plt.xlabel('x')
plt.ylabel('f(x) = x²')
plt.title('Function f(x) = x² on [0, 1]')
plt.grid(True, ls=':')
plt.show()

Our goal is to integrate this function over the interval, which can be interpreted as finding the area under the curve.

For the above function, it's easy enough to do this numerically. So, do so first to figure out what the answer should be before we integrate numerically. Please perform

$$
\int_0^1 dx~x^2
$$

by hand, and keep the result on-hand.

To perform this calculation numerically, we will implement the trapezoidal rule: split the interval [a,b] into `N` subintervals, evaluate `f` at the grid points, and approximate the integral by summing area of trapezoids. If you'd like a visual refresher of what the trapezoidal rule is doing, check out this video: https://www.youtube.com/watch?v=HVukQhDrNIQ

## Trapezoidal Rule

The trapezoidal rule approximates the integral of a function over an interval [a, b] by dividing the interval into N equal subintervals and approximating the area under the curve as a sum of trapezoids.

**Formula:**
$$
\int_a^b f(x)\,dx \approx \frac{h}{2}[f(x_0) + 2f(x_1) + 2f(x_2) + \cdots + 2f(x_{N-1}) + f(x_N)] = \frac{h}{2}(f(x_0) + f(x_N)) + h\sum_{i=1}^{N-1} f(x_i)
$$

where:
- $h = \frac{b-a}{N}$ is the width of each subinterval
- $x_i = a + ih$ for $i = 0, 1, \ldots, N$ are the grid points
- The first and last function values are weighted by $\frac{1}{2}$, while interior values are weighted by 1

If you don't understand the formula, please ask questions.

Because this method is an approximation from the real result, it will contain some error. This is how much the approximation differs from the exact result.

**Error Behavior:**
For smooth functions, the local error per subinterval is $O(h^3)$, leading to a global error of $O(h^2) = O(1/N^2)$. This means the error decreases quadratically as the number of subintervals increases.

Your job will be to implement the above formula into Python code. Take careful note of the indices above.

In [ ]:
def trapz_integral(f, a, b, N):
    """Approximate integral of f from a to b using the trapezoidal rule with N subintervals.
    f : callable accepting numpy arrays or floats
    a, b : floats (interval endpoints)
    N : int (number of subintervals)
    Returns float.
    """
    x = np.linspace(a, b, N+1)
    y = f(x)
    h = (b - a) / N

    result = ???

    return result

Now, test your implementation of the trapezoidal rule. First, define the function:

In [ ]:
def f(x):
    return ???

Next, define your interval.

In [ ]:
a, b = ???

Now define what the analytical result is (the result of your integration by hand):

In [ ]:
analytic = ???

In [ ]:
Ns = [10, 100, 1000, 5000]
results = []
errors = []
for N in Ns:
    val = trapz_integral(f, a, b, N)
    err = abs(val - analytic)
    results.append(val)
    errors.append(err)
    print(f"N={N:5d}  trapz={val:.12f}  error={err:.3e}")

# Plot convergence
plt.figure(figsize=(6,4))
plt.loglog(Ns, errors, '-o')
plt.xlabel('N (number of subintervals)')
plt.ylabel('Absolute error')
plt.title('Trapezoidal rule convergence for x**2 on [0,1]')
plt.grid(True, which='both', ls=':')
plt.show()

By increasing N (the number of subintervals), the error on the calculation decreases.

### Compare with SciPy's `quad`
`scipy.integrate.quad` is an adaptive algorithm (uses an error estimate and refines). We'll call it and compare results and runtime characteristics.

In [ ]:
quad_val, quad_err = integrate.quad(f, a, b)
print(f"scipy.integrate.quad value = {quad_val:.15f} (reported error {quad_err:.3e})")
print(f"Analytic value = {analytic:.15f}, abs error = {abs(quad_val-analytic):.3e}")

**Reflection:**
- Is there anything stopping your calculation from getting as precise as the analytical solution? Can the error decrease to be arbitrarily small?

## Section 2 — Integrating the NFW Profile

**Make sure you understand each step in this section.**

### Understanding the Spherical Jacobian

When we integrate a density function in spherical coordinates, we need to account for how volume elements change shape in different coordinate systems.

### What is a Jacobian?

A **Jacobian** is a correction factor that accounts for how volume (or area) changes when we transform from one coordinate system to another.

In Cartesian coordinates $(x, y, z)$, a tiny volume element is simply:
$$dV = dx\,dy\,dz$$

But in **spherical coordinates** $(r, \theta, \phi)$, where:
- $r$ is the radial distance from the origin
- $\theta$ is the polar angle (from the z-axis)
- $\phi$ is the azimuthal angle (around the z-axis)

The volume element is **not** just $dr\,d\theta\,d\phi$. Instead, it is:
$$dV = r^2 \sin\theta\, dr\,d\theta\,d\phi$$

The factor $r^2 \sin\theta$ is the **Jacobian** for spherical coordinates.

### Why Do We Need It?

Imagine drawing a small "box" in spherical coordinates. As you move farther from the origin (larger $r$), the physical size of that box grows, even though $dr$, $d\theta$, and $d\phi$ stay the same. The Jacobian $r^2\sin\theta$ corrects for this stretching.

### Integrating the NFW Density Profile

To find the total dark matter mass in the Milky Way, we integrate the density $\rho(r)$ over a spherical volume:
$$M_\mathrm{dm} = \int \rho(r)\, dV = \int_0^{r_\mathrm{max}} \int_0^\pi \int_0^{2\pi} \rho(r) \cdot r^2 \sin\theta\, d\phi\, d\theta\, dr$$

Because the NFW profile is **spherically symmetric** (depends only on $r$, not on $\theta$ or $\phi$), the angular integrals simplify:
$$\int_0^{2\pi} d\phi = 2\pi, \quad \int_0^\pi \sin\theta\, d\theta = 2$$

So the mass becomes:
$$M_\mathrm{dm} = 4\pi \int_0^{r_\mathrm{max}} \rho(r)\, r^2\, dr$$

The $r^2$ factor (part of the Jacobian) is **essential** — without it, we'd be calculating a quantity without physical interpretation. Rebuke this blasphemy whenever you notice it.

### Analytical Calculation

The amount of dark matter in the Milky Way, $M_\mathrm{dm}$, is given by

$$
M_\mathrm{dm} = 4\pi \int_0^{r_\mathrm{max}} dr~ \frac{\rho_0 r^2}{\frac{r}{R_s}\left(1 + \frac{r}{R_s}\right)^2}
$$

Let's perform a change of variables to make this integral look cleaner. Let's define
$$
x \equiv \frac{r}{R_s} \rightarrow r = x R_s \text{ and } dr = R_s dx
$$
so that the integral now reads

$$
M_\mathrm{dm} = 4\pi \int_0^{x_\mathrm{max}} dx~ \frac{\rho_0 x^2 R_s^3}{x\left(1 + x\right)^2}
$$

Now, we can factor out anything that's not a function of $x$ and combine like terms:

$$
M_\mathrm{dm} = 4\pi\rho_0 R_s^3 \int_0^{x_\mathrm{max}} dx~ \frac{x}{\left(1 + x\right)^2}
$$

Question for you: What are the units of this expression? Make sure you understand what each quantity is and which units it has. What are the units of the prefactor (the stuff outside the integral?) If you calculate $4\pi_0 R_s^3$ with $\rho_0 = 0.3\text{GeV cm}^{-3}$ and $R_s = 20~\text{kpc}$, what do you get? Put this number in reasonable units (how many solar masses)? We'll call this the *scale mass* of the Milky Way:

$$
M_\mathrm{dm} = M_\mathrm{scale} \int_0^{x_\mathrm{max}} dx~ \frac{x}{\left(1 + x\right)^2}
$$

We have successfully *non-dimensionalized* our problem. This means we have hidden away any units or numbers that distract from the behavior of the function. Luckily, this function is simple enough to be integrated analytically (manually). This will be a side task for you. The rest of this notebook will focus on numerically calculating the integral. The last ingredient is the actual value for $x_\mathrm{max}$.

Using the definition of $x$, $x_\mathrm{max} = \frac{r_\mathrm{max}}{R_s}$. You might think our maximum radius would be the radius of the Milky Way, i.e., $r_\mathrm{max} = R_s = 20~$kpc. However, $R_s$ is the radius of the *visible* part of the Milky Way. The radius of the Dark Matter Halo goes way further, like 200 kpc such that $x_\mathrm{max} = 10$.

$$
M_\mathrm{dm} = M_\mathrm{scale} \int_0^{10} dx~ \frac{x}{\left(1 + x\right)^2}
$$

We'll now move on to calculating $M_\mathrm{dm}$ numerically. First, set the value of the scale mass that you calculated above:

In [ ]:
M_scale = ??? # solar masses

Next, define the above non-dimensionalized integrand.

In [ ]:
# Define your integrand here
def nfw_density_nondim(x):
    """ NFW density function, non-dimensionalized."""
    return ???

Now set the integration limits.

In [ ]:
# Choose interval
a, b = ???, ???

Finally, complete the calculation using <code>quad</code>.

In [ ]:
# Compute numerical integrals
quad_val, quad_err = integrate.quad(nfw_density_nondim, a, b)

print(f"integrand = {quad_val:.12f} (reported err {quad_err:.3e})")

Now, putting everything together, the amount of dark matter in the Milky Way Galaxy assuming a NFW Density Profile is

In [ ]:
M_dm = M_scale * quad_val
print(f"M_dm = {M_dm:.3e} solar masses")